# NB_MDP_30 · Gold (un objeto)

Lo llama `NB_MDP_00_orquestador` después de silver, respetando `DEPENDS_ON_GOLD`.

| `LOAD_MODE` | Qué hace con `SQL_TEXT` |
|---|---|
| `OVERWRITE` | Ejecuta el `SELECT` y reemplaza la tabla. Para dimensiones chicas o agregados |
| `MERGE` | Ejecuta el `SELECT` y hace upsert por `MERGE_KEY`. Para hechos grandes |
| `MLV` | Ejecuta el `CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS ...`. La primera vez crea la vista; después Fabric la refresca según la programación del lineage de vistas materializadas |

In [ ]:
# Parámetros
run_id = "manual"
gold_json = "{}"

In [ ]:
import json
from pyspark.sql import functions as F
from delta.tables import DeltaTable

g = json.loads(gold_json)
resultado = {"layer": "gold", "object_code": g.get("gold_code"), "status": None,
             "rows_read": None, "rows_written": None, "error": None}


def filas_escritas(tabla):
    m = DeltaTable.forName(spark, tabla).history(1).select("operationMetrics").first()[0] or {}
    if "numTargetRowsInserted" in m:   # MERGE: numOutputRows incluye filas reescritas sin cambios
        return int(m["numTargetRowsInserted"]) + int(m.get("numTargetRowsUpdated", 0))
    return int(m.get("numOutputRows", 0))


try:
    destino, modo = g["target_table"], g["load_mode"]
    if modo == "MLV":
        spark.sql(g["sql_text"])
        resultado["status"] = "SUCCEEDED"
    else:
        df = spark.sql(g["sql_text"]).withColumn("_gold_updated_at", F.current_timestamp())
        if modo == "OVERWRITE" or not spark.catalog.tableExists(destino):
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
        else:
            claves = [k.strip() for k in g["merge_key"].split(",") if k.strip()]
            cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in claves)
            spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
            (DeltaTable.forName(spark, destino).alias("t")
                .merge(df.alias("s"), cond)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
        resultado.update(status="SUCCEEDED", rows_written=filas_escritas(destino))
except Exception as ex:
    resultado.update(status="FAILED", error=f"{type(ex).__name__}: {ex}"[:3000])

print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
notebookutils.notebook.exit(json.dumps(resultado, ensure_ascii=False, default=str))